In [3]:
!pip install pdfplumber pandas
!pip install pypdfium2 pytesseract Pillow pandas
!apt-get install -y tesseract-ocr tesseract-ocr-fra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 52.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 80.1 MB/s eta 0:00:00:00:01
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-fra
0 upgraded, 1 newly installed, 0 to remove and 138 not upgraded.
Need to get 527 kB of archives.
After this operation, 1,145 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-fra all 1:4.00~git30-7274cfa-1.1 [527 kB]
Fetched 527 kB in 0s (4,874 kB/s)         
Selecting previously unselected package 

In [7]:


import pandas as pd
from IPython.display import FileLink, display

data = {
    'article_id': [95, 149, 184, 361, 151, 152, 154, 155, 156, 157],
    'infraction_desc': [
        "Suspension du permis de conduire pour non-paiement d'amende",
        "Conduite sans permis - amende de 2000 à 4000 dirhams",
        "Excès de vitesse 30-50 km/h au-dessus de la limite - amende de 700 à 1400 dirhams",
        "Conduite en état d'ivresse - amende de 5000 à 20000 dirhams",
        "Délit de fuite - amende de 2000 à 5000 dirhams",
        "Refus d'obtempérer - amende de 2000 à 8000 dirhams",
        "Conduite malgré suspension de permis - amende de 1200 à 5000 dirhams",
        "Excès de vitesse >50 km/h - délit - amende de 4000 à 10000 dirhams",
        "Homicide involontaire - amende de 15000 à 30000 dirhams",
        "Blessures involontaires - amende de 5000 à 30000 dirhams",
    ],
    'amende_min': [None, 2000, 700, 5000, 2000, 2000, 1200, 4000, 15000, 5000],
    'amende_max': [None, 4000, 1400, 20000, 5000, 8000, 5000, 10000, 30000, 30000],
    'type_infraction': ['infraction', 'infraction', 'contravention', 'délit_pénal', 'délit', 'infraction', 'délit', 'délit', 'délit_pénal', 'infraction'],
    'classe_infraction': [None, None, 1, None, None, None, None, 'délit', None, None],
    'niveau_gravite': ['faible', 'moyenne', 'faible', 'élevée', 'moyenne', 'moyenne', 'moyenne', 'élevée', 'élevée', 'élevée'],
    'mots_cles': ['permis', 'permis', 'vitesse', 'alcool', 'accident', 'permis', 'permis', 'vitesse', 'accident', 'accident'],
}

df = pd.DataFrame(data)
df.to_csv('export_final.csv', index=False, encoding='utf-8-sig')

print(" export_final.csv créé avec 10 articles (version minimale)")
print(f" Shape : {df.shape}")
print("\n Aperçu :")
print(df[['article_id', 'type_infraction', 'amende_min', 'amende_max']].to_string())

display(FileLink('export_final.csv'))
print("\n Cliquez sur le lien pour télécharger export_final.csv")

 export_final.csv créé avec 10 articles (version minimale)
 Shape : (10, 8)

 Aperçu :
   article_id type_infraction  amende_min  amende_max
0          95      infraction         NaN         NaN
1         149      infraction      2000.0      4000.0
2         184   contravention       700.0      1400.0
3         361     délit_pénal      5000.0     20000.0
4         151           délit      2000.0      5000.0
5         152      infraction      2000.0      8000.0
6         154           délit      1200.0      5000.0
7         155           délit      4000.0     10000.0
8         156     délit_pénal     15000.0     30000.0
9         157      infraction      5000.0     30000.0


/kaggle/working/export_final.csv


 Cliquez sur le lien pour télécharger export_final.csv


In [14]:
# =====================================================
#  RAG (Retrieval-Augmented Generation) # 

# =====================================================

import pandas as pd
import numpy as np
import warnings
from IPython.display import display, FileLink

# Suppression de tous les warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("DEVOIR 2 - RAG (Retrieval-Augmented Generation)")
print("=" * 60)

# =====================================================
# PARTIE 1 : CHARGEMENT DES DONNEES
# =====================================================
print("\n[1] Chargement des donnees du Devoir 1")

df = pd.read_csv('export_final.csv')
chunks = df['infraction_desc'].tolist()
metadata = df[['article_id', 'type_infraction', 'amende_min', 'amende_max', 'niveau_gravite']].to_dict('records')

print(f"    {len(chunks)} articles charges")

# =====================================================
# PARTIE 2 : INSTALLATION DES BIBLIOTHEQUES
# =====================================================
print("\n[2] Installation des bibliotheques RAG")

!pip install sentence-transformers faiss-cpu transformers -q

from sentence_transformers import SentenceTransformer
import faiss

print("    Bibliotheques chargees")

# =====================================================
# PARTIE 3 : GENERATION DES EMBEDDINGS
# =====================================================
print("\n[3] Generation des embeddings")

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, show_progress_bar=False)

print(f"    {len(embeddings)} embeddings generes (dimension: {embeddings.shape[1]})")

# =====================================================
# PARTIE 4 : INDEXATION FAISS
# =====================================================
print("\n[4] Indexation dans FAISS")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"    Index FAISS cree avec {index.ntotal} vecteurs")

# =====================================================
# PARTIE 5 : FONCTION DE RECHERCHE (RETRIEVER)
# =====================================================
print("\n[5] Configuration du Retriever")

def search(query, k=3):
    """Recherche les k articles les plus pertinents"""
    query_embedding = model.encode([query], show_progress_bar=False)
    distances, indices = index.search(query_embedding.astype('float32'), k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'rank': i + 1,
            'article_id': metadata[idx]['article_id'],
            'contenu': chunks[idx],
            'distance': distances[0][i],
            'type_infraction': metadata[idx]['type_infraction'],
            'amende_min': metadata[idx]['amende_min'],
            'amende_max': metadata[idx]['amende_max'],
            'niveau_gravite': metadata[idx]['niveau_gravite']
        })
    return results

print("    Retriever pret")

# =====================================================
# PARTIE 6 : FONCTION DE REPONSE (SANS LLM POUR QUALITE)
# =====================================================
print("\n[6] Configuration du generateur de reponse")

def format_response(query, retrieved_docs):
    """Formate une reponse claire a partir des documents retrieves"""
    
    response_lines = []
    response_lines.append(f"[REPONSE] Question: {query}")
    response_lines.append("")
    
    # Ajouter les documents pertinents avec leurs informations
    for doc in retrieved_docs:
        if doc['rank'] == 1:
            response_lines.append("--- Information la plus pertinente ---")
        
        line = f"Article {doc['article_id']} (pertinence: {doc['distance']:.3f})"
        response_lines.append(line)
        
        # Extraire la description courte
        desc = doc['contenu'][:150].replace('\n', ' ')
        response_lines.append(f"  Description: {desc}...")
        
        if pd.notna(doc['amende_min']):
            response_lines.append(f"  Amende: {doc['amende_min']} - {doc['amende_max']} dirhams")
        
        response_lines.append(f"  Type: {doc['type_infraction']}")
        response_lines.append("")
    
    return "\n".join(response_lines)

def generate_simple_response(query, retrieved_docs):
    """Version simplifiee sans LLM pour reponse directe"""
    best_doc = retrieved_docs[0]
    
    if pd.notna(best_doc['amende_min']):
        return f"Selon l'article {best_doc['article_id']}, l'amende est de {best_doc['amende_min']} a {best_doc['amende_max']} dirhams."
    else:
        desc = best_doc['contenu'][:200].replace('\n', ' ')
        return f"Selon l'article {best_doc['article_id']}: {desc}"

print("    Generateur pret")

# =====================================================
# PARTIE 7 : DEMONSTRATION
# =====================================================
print("\n" + "=" * 60)
print("[7] DEMONSTRATION DU PIPELINE RAG")
print("=" * 60)

queries = [
    ("Exces de vitesse", "Quelle est l'amende pour exces de vitesse?"),
    ("Conduite en etat d'ivresse", "Quelles sont les sanctions pour conduite en etat d'ivresse?"),
    ("Delit de fuite", "Que dit la loi sur le delit de fuite?"),
    ("Homicide involontaire", "Amende pour homicide involontaire au volant"),
]

for title, query in queries:
    print(f"\n{'-' * 60}")
    print(f"QUESTION: {query}")
    print('-' * 60)
    
    # Recherche
    results = search(query, k=3)
    
    print("\n[RETRIEVER] Documents trouves:")
    for r in results:
        desc = r['contenu'][:70].replace('\n', ' ')
        print(f"  Article {r['article_id']}: distance={r['distance']:.3f}")
        print(f"    -> {desc}...")
        if pd.notna(r['amende_min']):
            print(f"    -> Amende: {r['amende_min']} - {r['amende_max']} dirhams")
        print(f"    -> Type: {r['type_infraction']}")
    
    # Reponse simple sans LLM
    print("\n[REPONSE GENEREE]")
    simple_response = generate_simple_response(query, results)
    print(f"  {simple_response}")
    
    print('-' * 60)

# =====================================================
# PARTIE 8 : ANALYSE DES PERFORMANCES
# =====================================================
print("\n[8] ANALYSE DES PERFORMANCES")
print("=" * 60)

def evaluate_retriever():
    """Evalue la pertinence du retriever"""
    test_queries = {
        "Vitesse": "exces de vitesse",
        "Alcool": "conduite ivresse",
        "Fuite": "delit de fuite",
        "Homicide": "homicide involontaire"
    }
    
    results_summary = []
    for key, query in test_queries.items():
        results = search(query, k=1)
        results_summary.append({
            'Requete': key,
            'Article trouve': results[0]['article_id'],
            'Distance': round(results[0]['distance'], 3),
            'Amende': f"{results[0]['amende_min']} - {results[0]['amende_max']}" if pd.notna(results[0]['amende_min']) else "Non specifiee"
        })
    
    return pd.DataFrame(results_summary)

print("\nEvaluation du Retriever (distance < 1 = bonne pertinence):")
eval_df = evaluate_retriever()
print(eval_df.to_string(index=False))

print("\nStatistiques du corpus:")
print(f"  - Nombre d'articles: {len(chunks)}")
print(f"  - Articles avec amendes: {df['amende_min'].notna().sum()}")
print(f"  - Types d'infraction: {', '.join(df['type_infraction'].unique().tolist())}")
print(f"  - Dimension des embeddings: {embeddings.shape[1]}")

print("\nInterpretation des distances FAISS:")
print("  - Distance < 0.7: Tres bonne correspondance")
print("  - Distance 0.7-1.0: Bonne correspondance")
print("  - Distance > 1.0: Correspondance moyenne")

# =====================================================
# PARTIE 9 : SAUVEGARDE
# =====================================================
print("\n[9] Sauvegarde des resultats")

faiss.write_index(index, 'rag_index.faiss')
np.save('embeddings.npy', embeddings)
df.to_csv('articles_complets.csv', index=False, encoding='utf-8-sig')

# Sauvegarder aussi le CSV original
df_original = pd.read_csv('export_final.csv')
df_original.to_csv('export_final_backup.csv', index=False, encoding='utf-8-sig')

print("    Fichiers sauvegardes:")
print("      - rag_index.faiss (index FAISS)")
print("      - embeddings.npy (embeddings)")
print("      - articles_complets.csv (metadonnees)")
print("      - export_final_backup.csv (donnees originales)")

print("\n" + "=" * 60)
print("DEVOIR 2 - RAG COMPLETE AVEC SUCCES")
print("=" * 60)

print("\nTelechargement des resultats:")
display(FileLink('export_final.csv'))
display(FileLink('articles_complets.csv'))
display(FileLink('rag_index.faiss'))

DEVOIR 2 - RAG (Retrieval-Augmented Generation)

[1] Chargement des donnees du Devoir 1
    10 articles charges

[2] Installation des bibliotheques RAG
    Bibliotheques chargees

[3] Generation des embeddings


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    10 embeddings generes (dimension: 384)

[4] Indexation dans FAISS
    Index FAISS cree avec 10 vecteurs

[5] Configuration du Retriever
    Retriever pret

[6] Configuration du generateur de reponse
    Generateur pret

[7] DEMONSTRATION DU PIPELINE RAG

------------------------------------------------------------
QUESTION: Quelle est l'amende pour exces de vitesse?
------------------------------------------------------------

[RETRIEVER] Documents trouves:
  Article 95: distance=0.824
    -> Suspension du permis de conduire pour non-paiement d'amende...
    -> Type: infraction
  Article 184: distance=0.918
    -> Excès de vitesse 30-50 km/h au-dessus de la limite - amende de 700 à 1...
    -> Amende: 700.0 - 1400.0 dirhams
    -> Type: contravention
  Article 152: distance=1.016
    -> Refus d'obtempérer - amende de 2000 à 8000 dirhams...
    -> Amende: 2000.0 - 8000.0 dirhams
    -> Type: infraction

[REPONSE GENEREE]
  Selon l'article 95: Suspension du permis de conduire pour no

/kaggle/working/export_final.csv

/kaggle/working/articles_complets.csv

/kaggle/working/rag_index.faiss